# Phase 0 Empirical Data Validation — United Airlines Standby Opportunity Analytics (`ORD` Hub)

> **Disclaimers:**
> - The **Standby Opportunity Index (`soi_historical` & `soi_planning`)** is a **0–100 relative heuristic** built from public aviation datasets (BTS, FAA, TSA) and curated fleet references — **NOT a probability of boarding**.
> - This project is **not affiliated with, endorsed by, or sponsored by United Airlines, Inc.**

This notebook executes the **Phase 0 Validation** across all five public data sources stored in `../data/raw/` (`2019-01..2019-12` and `2023-01..2026-08`) and reproduces all five empirical tables in `docs/phase0_validation_report.md`:
1. **Dataset Inventory, Granularity, Coverage & Null Rates**
2. **Field-by-Field Verification & Reporting vs. Marketing Carrier OTP Discrepancy Proof**
3. **OTP Performed Flights vs. BTS T-100 `DEPARTURES_PERFORMED` Reconciliation (Tables 3A & 3B)**
4. **OTP `Tail_Number` → FAA Aircraft Registry (`MASTER.txt` + `DEREG.txt` → `ACFTREF.txt`) Match Rate (Tables 4A & 4B)**
5. **Empirical Audit of All 6 Known Data Quirks (+ 3 Structural Quirks) with Verbatim Examples**

In [1]:
import collections
import csv
import hashlib
import io
import json
import statistics
from pathlib import Path

RAW_DIR = Path("/home/antigravity/united-standby-analytics/data/raw")
LOOKUPS_DIR = RAW_DIR / "lookups"
TSA_DIR = RAW_DIR / "tsa"
FAA_DIR = RAW_DIR / "faa"
T100_DIR = RAW_DIR / "t100"
OTP_DIR = RAW_DIR / "otp"

otp_mkt_files = sorted(OTP_DIR.glob("otp_mkt_ord_ua_*.csv"))
t100_files = sorted(T100_DIR.glob("t100_segment_all_*.csv"))
lookup_files = sorted(LOOKUPS_DIR.glob("*.csv"))
faa_files = sorted(FAA_DIR.glob("*.txt"))
tsa_file = TSA_DIR / "tsa_checkpoint_volumes.csv"

print("=== Phase 0 Raw Data Inventory in data/raw/ ===")
print(f"1. BTS Marketing Carrier OTP monthly ORD-UA files : {len(otp_mkt_files)} files ({otp_mkt_files[0].name} .. {otp_mkt_files[-1].name})")
print(f"2. BTS Reporting Carrier OTP sample (2024_01)     : {(OTP_DIR / 'otp_rep_ord_2024_01.csv').exists()}")
print(f"3. BTS T-100 Segment annual files (FMG)           : {len(t100_files)} files ({[f.name for f in t100_files]})")
print(f"4. BTS Lookup CSV tables                          : {len(lookup_files)} files ({[f.name for f in lookup_files]})")
print(f"5. FAA Aircraft Registry files                    : {len(faa_files)} files ({[f.name for f in faa_files]})")
print(f"6. TSA Checkpoint Volumes CSV                     : {tsa_file.name} ({tsa_file.stat().st_size:,} bytes)")

=== Phase 0 Raw Data Inventory in data/raw/ ===
1. BTS Marketing Carrier OTP monthly ORD-UA files : 56 files (otp_mkt_ord_ua_2019_01.csv .. otp_mkt_ord_ua_2026_08.csv)
2. BTS Reporting Carrier OTP sample (2024_01)     : True
3. BTS T-100 Segment annual files (FMG)           : 5 files (['t100_segment_all_2019.csv', 't100_segment_all_2023.csv', 't100_segment_all_2024.csv', 't100_segment_all_2025.csv', 't100_segment_all_2026.csv'])
4. BTS Lookup CSV tables                          : 5 files (['L_AIRCRAFT_TYPE.csv', 'L_AIRPORT.csv', 'L_AIRPORT_ID.csv', 'L_CARRIER_HISTORY.csv', 'L_UNIQUE_CARRIERS.csv'])
5. FAA Aircraft Registry files                    : 3 files (['ACFTREF.txt', 'DEREG.txt', 'MASTER.txt'])
6. TSA Checkpoint Volumes CSV                     : tsa_checkpoint_volumes.csv (131,744 bytes)


In [2]:
# Table 2: Compare BTS Reporting Carrier OTP (2024-01) vs. BTS Marketing Carrier OTP (2024-01) at ORD
with open(OTP_DIR / "otp_rep_ord_2024_01.csv", "r", encoding="utf-8") as f:
    rep_reader = csv.reader(f)
    rep_raw_hdr = next(rep_reader)
    rep_clean_hdr = [c.strip() for c in rep_raw_hdr if c.strip()]
    rep_c_idx = rep_clean_hdr.index("Reporting_Airline")
    rep_counts = collections.Counter(r[rep_c_idx] for r in rep_reader if r)

with open(OTP_DIR / "otp_mkt_ord_ua_2024_01.csv", "r", encoding="utf-8") as f:
    mkt_reader = csv.reader(f)
    mkt_hdr = next(mkt_reader)
    mkt_op_idx = mkt_hdr.index("Operating_Airline")
    mkt_counts = collections.Counter(r[mkt_op_idx] for r in mkt_reader if r)

with open(RAW_DIR / "otp_manifest_log.csv", "r", encoding="utf-8") as f:
    manifest_rows = list(csv.DictReader(f))

print("=== Reporting Carrier OTP vs. Marketing Carrier OTP (2024-01 ORD) ===")
print(f"Reporting OTP Raw Columns : {len(rep_raw_hdr)} ({len(rep_clean_hdr)} non-empty)")
print(f"  Has Marketing_Airline_Network / MKT_UNIQUE_CARRIER? {'Marketing_Airline_Network' in rep_clean_hdr or 'MKT_UNIQUE_CARRIER' in rep_clean_hdr}")
print(f"  Has Operating_Airline / OP_UNIQUE_CARRIER?          {'Operating_Airline' in rep_clean_hdr or 'OP_UNIQUE_CARRIER' in rep_clean_hdr}")
print(f"Marketing OTP Raw Columns : {manifest_rows[0]['raw_col_count']} ({manifest_rows[0]['clean_col_count']} non-empty)")
print(f"  PREZIP raw header has trailing space 'Operating_Airline '? {all(r['has_operating_airline_trailing_space'] == 'True' for r in manifest_rows)} (across all {len(manifest_rows)} months)")
print()
print("2024-01 ORD Carrier Flight Counts (Reporting OTP vs. Marketing OTP MKT='UA'):")
for carrier in ["UA", "OO", "YX", "G7"]:
    r_cnt = rep_counts.get(carrier, 0)
    m_cnt = mkt_counts.get(carrier, 0)
    print(f"  {carrier}: Reporting OTP = {r_cnt:6,d} | Marketing OTP (MKT='UA') = {m_cnt:6,d} | Diff = {r_cnt - m_cnt:+6,d}")

=== Reporting Carrier OTP vs. Marketing Carrier OTP (2024-01 ORD) ===
Reporting OTP Raw Columns : 110 (109 non-empty)
  Has Marketing_Airline_Network / MKT_UNIQUE_CARRIER? False
  Has Operating_Airline / OP_UNIQUE_CARRIER?          False
Marketing OTP Raw Columns : 120 (119 non-empty)
  PREZIP raw header has trailing space 'Operating_Airline '? True (across all 56 months)

2024-01 ORD Carrier Flight Counts (Reporting OTP vs. Marketing OTP MKT='UA'):
  UA: Reporting OTP =  6,679 | Marketing OTP (MKT='UA') =  6,679 | Diff =     +0
  OO: Reporting OTP =  4,115 | Marketing OTP (MKT='UA') =  3,591 | Diff =   +524
  YX: Reporting OTP =  1,376 | Marketing OTP (MKT='UA') =    776 | Diff =   +600
  G7: Reporting OTP =      0 | Marketing OTP (MKT='UA') =    672 | Diff =   -672


In [3]:
# Table 1: Profile all 56 months of BTS Marketing Carrier OTP (2019-01..2019-12 & 2023-01..2026-08)
SUMMARY_PATH = Path("/home/antigravity/united-standby-analytics/notebooks/phase0_empirical_summary.json") if Path("/home/antigravity/united-standby-analytics/notebooks/phase0_empirical_summary.json").exists() else Path("phase0_empirical_summary.json")
emp = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
otp_info = emp["otp"]

print("=== BTS Marketing Carrier OTP 56-Month Summary (ORD, Marketing_Airline_Network='UA', Duplicate='N') ===")
print(f"Months Verified          : {otp_info['months_count']} ({otp_info['date_min']} .. {otp_info['date_max']})")
print(f"Total Raw US Flight Rows : {otp_info['total_us_raw_rows']:,}")
print(f"Total ORD All-Carrier    : {otp_info['total_ord_all_rows']:,}")
print(f"Total ORD UA-Marketed    : {otp_info['total_ord_ua_rows']:,}")
print(f"  Performed (Cancelled=0): {otp_info['total_performed']:,} ({otp_info['total_performed']/otp_info['total_ord_ua_rows']*100:.2f}%)")
print(f"  Cancelled (Cancelled=1): {otp_info['total_cancelled']:,} ({otp_info['total_cancelled']/otp_info['total_ord_ua_rows']*100:.2f}%)")
print(f"  Diverted  (Diverted=1) : {otp_info['total_diverted']:,} ({otp_info['total_diverted']/otp_info['total_ord_ua_rows']*100:.2f}%)")
print()
print("By Year Breakdown (Scheduled Flights & Operating Carriers at ORD):")
for yr, yd in otp_info["by_year"].items():
    c_str = ", ".join(f"{k}:{v:,}" for k, v in yd["carriers"].items())
    print(f"  {yr} ({yd['months']:2d} mos): US={yd['us_raw_rows']:9,d} | ORD_UA={yd['ord_ua_rows']:7,d} (Perf={yd['performed']:7,d}, Canc={yd['cancelled']:5,d}) | Carriers: {c_str}")

print("\nNull Rates Across 830,941 ORD UA Rows (Overall vs. Performed vs. Cancelled vs. Diverted):")
for col in ["FlightDate", "Marketing_Airline_Network", "Operating_Airline", "Origin", "Dest", "CRSDepTime", "Tail_Number", "DepTime", "DepDelay", "ArrTime", "ArrDelay", "ActualElapsedTime", "CancellationCode"]:
    n_all = otp_info["null_overall"].get(col, 0)
    n_perf = otp_info["null_on_performed"].get(col, 0)
    n_canc = otp_info["null_on_cancelled"].get(col, 0)
    n_div = otp_info["null_on_diverted"].get(col, 0)
    print(f"  {col:26s}: Overall={n_all:7,d} ({n_all/otp_info['total_ord_ua_rows']*100:6.2f}%) | Performed={n_perf:7,d} | Cancelled={n_canc:6,d} | Diverted={n_div:5,d}")

=== BTS Marketing Carrier OTP 56-Month Summary (ORD, Marketing_Airline_Network='UA', Duplicate='N') ===
Months Verified          : 56 (2019-01 .. 2026-08)
Total Raw US Flight Rows : 35,881,885
Total ORD All-Carrier    : 1,637,491
Total ORD UA-Marketed    : 830,941
  Performed (Cancelled=0): 814,847 (98.06%)
  Cancelled (Cancelled=1): 16,094 (1.94%)
  Diverted  (Diverted=1) : 2,390 (0.29%)

By Year Breakdown (Scheduled Flights & Operating Carriers at ORD):
  2019 (12 mos): US=8,091,684 | ORD_UA=199,882 (Perf=194,127, Canc=5,755) | Carriers: UA:77,660, ZW:38,321, OO:34,022, YX:13,697, EV:13,073, AX:12,774, G7:10,247, C5:88
  2023 (12 mos): US=7,278,739 | ORD_UA=151,447 (Perf=149,486, Canc=1,961) | Carriers: UA:84,539, OO:43,274, ZW:8,464, YX:8,451, G7:6,719
  2024 (12 mos): US=7,546,968 | ORD_UA=161,074 (Perf=158,869, Canc=2,205) | Carriers: UA:89,830, OO:48,968, YX:11,748, G7:10,528
  2025 (12 mos): US=7,736,770 | ORD_UA=181,927 (Perf=179,482, Canc=2,445) | Carriers: UA:95,572, OO:49,66

In [4]:
# Table 1 & Sanity Check #4: Profile BTS T-100 Segment (All Carriers, 2019 & 2023-2026) and System-Wide UA Load Factor
t100_info = emp["t100"]
print("=== BTS T-100 Segment All Carriers (FMG: Domestic + International) ===")
print(f"Total Raw Segment Rows (5 years) : {t100_info['total_raw_rows']:,} ({t100_info['col_count']} columns)")
print(f"Total ORD CLASS='F' Rows         : {t100_info['total_ord_class_f_rows']:,}")
print(f"  Domestic (DU/DF)               : {t100_info['total_ord_class_f_dom_rows']:,}")
print(f"  International (IU/IF)          : {t100_info['total_ord_class_f_intl_rows']:,}")
print(f"Null Counts on Key ORD Columns   : {t100_info['null_counts_ord_f']} (0.00% null across all required fields)")
print()
print("Annual T-100 Coverage & Sanity Check #4 (System-Wide UA Load Factor Target ~80-85%):")
for yr, td in t100_info["by_year"].items():
    m_range = f"{min(td['months_available']):02d}..{max(td['months_available']):02d}"
    print(f"  {yr} (Months {m_range}): Raw={td['raw_rows']:7,d} | ORD_F={td['ord_class_f_rows']:6,d} | UA Sys Deps={td['ua_system_deps']:9,d} | UA Sys Seats={td['ua_system_seats']:11,d} | UA Sys Pax={td['ua_system_pax']:11,d} | UA Sys LF={td['ua_system_lf_pct']:.2f}%")

=== BTS T-100 Segment All Carriers (FMG: Domestic + International) ===
Total Raw Segment Rows (5 years) : 2,520,139 (50 columns)
Total ORD CLASS='F' Rows         : 75,863
  Domestic (DU/DF)               : 65,786
  International (IU/IF)          : 10,077
Null Counts on Key ORD Columns   : {} (0.00% null across all required fields)

Annual T-100 Coverage & Sanity Check #4 (System-Wide UA Load Factor Target ~80-85%):
  2019 (Months 01..12): Raw=487,923 | ORD_F=16,117 | UA Sys Deps=  795,002 | UA Sys Seats=138,567,871 | UA Sys Pax=116,626,076 | UA Sys LF=84.17%
  2023 (Months 01..12): Raw=529,562 | ORD_F=14,709 | UA Sys Deps=  909,307 | UA Sys Seats=160,218,153 | UA Sys Pax=134,394,576 | UA Sys LF=83.88%
  2024 (Months 01..12): Raw=549,799 | ORD_F=15,901 | UA Sys Deps=  962,352 | UA Sys Seats=169,518,510 | UA Sys Pax=140,993,783 | UA Sys LF=83.17%
  2025 (Months 01..12): Raw=571,262 | ORD_F=17,395 | UA Sys Deps=1,012,639 | UA Sys Seats=179,418,829 | UA Sys Pax=146,214,658 | UA Sys LF=81.4

In [5]:
# Table 3A, Table 3B & Quirk 5: OTP vs. T-100 Departure Reconciliation & Multiple Aircraft Types per Route-Month
print("=== Table 3A: OTP Performed vs. T-100 Domestic Departures Performed at ORD by Year & Carrier ===")
print(f"{'Year':4s} | {'Carrier':7s} | {'OTP_Sched':>9s} | {'OTP_Perf':>9s} | {'T100_Match':>10s} | {'T100_DomTot':>11s} | {'T100_Intl':>9s} | {'MatchDiff%':>10s} | {'Within±10%':>12s} | {'MedDiff%':>8s}")
print("-" * 112)
for r in emp["reconciliation"]:
    w_str = f"{r['within_10pct_rate']:5.1f}% ({r['within_10pct_route_months']}/{r['matched_route_months']})"
    print(f"{r['year']:4d} | {r['op_carrier']:7s} | {r['otp_scheduled']:9,d} | {r['otp_performed']:9,d} | {r['t100_matched_route_deps']:10,d} | {r['t100_total_ord_dom_deps']:11,d} | {r['t100_total_ord_intl_deps']:9,d} | {r['matched_diff_pct']:+9.2f}% | {w_str:>12s} | {r['median_route_diff_pct']:+7.2f}%")

q5 = emp["quirks"]["quirk_5_multi_acft_by_year"]
print("\n=== Quirk 5: Route-Months with >1 BTS AIRCRAFT_TYPE in T-100 (ORD Domestic) ===")
for yr, d in q5.items():
    print(f"  {yr}: UA+Express Family = {d['ua_family_multi_acft']:4,d}/{d['ua_family_route_months']:4,d} ({d['ua_family_multi_pct']:.2f}%) | UA Mainline = {d['ua_mainline_multi_acft']:4,d}/{d['ua_mainline_route_months']:4,d} ({d['ua_mainline_multi_pct']:.2f}%)")
print(f"Overall UA+Express Family Multi-Aircraft Route-Months: {emp['quirks']['quirk_5_overall_ua_family_multi_acft']:,} / {emp['quirks']['quirk_5_overall_ua_family_route_months']:,} ({emp['quirks']['quirk_5_overall_ua_family_multi_pct']:.2f}%)")
print(f"Max Aircraft Types on Single Route-Month Example     : {emp['quirks']['quirk_5_max_example']}")

=== Table 3A: OTP Performed vs. T-100 Domestic Departures Performed at ORD by Year & Carrier ===
Year | Carrier | OTP_Sched |  OTP_Perf | T100_Match | T100_DomTot | T100_Intl | MatchDiff% |   Within±10% | MedDiff%
----------------------------------------------------------------------------------------------------------------
2019 | UA      |    77,660 |    76,855 |     76,987 |      77,019 |    10,939 |     +0.17% |  98.7% (884/896) |   +0.00%
2019 | ZW      |    38,321 |    36,811 |     36,751 |      36,767 |         0 |     -0.16% |  98.4% (808/821) |   +0.00%
2019 | OO      |    34,022 |    32,691 |     54,874 |      62,168 |     4,344 |    +67.86% |  62.5% (680/1088) |   +0.00%
2019 | YX      |    13,697 |    13,221 |     13,133 |      13,555 |       407 |     -0.67% |  97.2% (515/530) |   +0.00%
2019 | EV      |    13,073 |    12,408 |     12,482 |      12,562 |     1,646 |     +0.60% |  96.8% (600/620) |   +0.00%
2019 | AX      |    12,774 |    12,138 |     12,135 |      12,143 |

In [6]:
# Table 4A & Table 4B: OTP Tail_Number -> FAA Registry (MASTER.txt + DEREG.txt -> ACFTREF.txt) Match Rates
faa_m = emp["faa_matching"]
ov = faa_m["overall"]
print("=== Table 4: FAA Aircraft Registry Tail Match Summary (830,941 ORD UA Flights) ===")
print(f"Total Flights: {ov['total_flights']:,} | Non-Null Tail: {ov['non_null_tail']:,} (100.00% of {ov['performed_flights']:,} performed flights!)")
print(f"Tier 1 (MASTER.txt) : {ov['matched_master']:,} ({ov['matched_master_pct']:.2f}%)")
print(f"Tier 2 (DEREG.txt)  : +{ov['matched_dereg']:,} (+{ov['matched_dereg_pct']:.2f}%)")
print(f"Combined FAA Match  : {ov['combined_matched']:,} ({ov['combined_matched_pct']:.2f}%) | Unmatched: {ov['unmatched']:,} ({ov['unmatched_pct']:.2f}% across {ov['unique_unmatched_tails']} unique tails)")

print("\nBy Year Breakdown (Table 4A):")
for yr, b in faa_m["by_year"].items():
    print(f"  {yr}: NonNull={b['non_null_tail']:7,d} | MASTER={b['matched_master']:7,d} ({b['matched_master_pct']:6.2f}%) | DEREG=+{b['matched_dereg']:6,d} ({b['matched_dereg_pct']:5.2f}%) | Combined={b['combined_matched']:7,d} ({b['combined_matched_pct']:6.2f}%) | Unmatched={b['unmatched']:6,d} ({b['unique_unmatched_tails']} tails)")

print("\nBy Operating Carrier Breakdown (Table 4B):")
for op, b in faa_m["by_carrier"].items():
    print(f"  {op:2s}: NonNull={b['non_null_tail']:7,d} | MASTER={b['matched_master']:7,d} ({b['matched_master_pct']:6.2f}%) | DEREG=+{b['matched_dereg']:6,d} ({b['matched_dereg_pct']:5.2f}%) | Combined={b['combined_matched']:7,d} ({b['combined_matched_pct']:6.2f}%) | Unmatched={b['unmatched']:6,d} ({b['unique_unmatched_tails']} tails)")

=== Table 4: FAA Aircraft Registry Tail Match Summary (830,941 ORD UA Flights) ===
Total Flights: 830,941 | Non-Null Tail: 822,719 (100.00% of 814,847 performed flights!)
Tier 1 (MASTER.txt) : 788,180 (95.80%)
Tier 2 (DEREG.txt)  : +21,206 (+2.58%)
Combined FAA Match  : 809,386 (98.38%) | Unmatched: 13,333 (1.62% across 44 unique tails)

By Year Breakdown (Table 4A):
  2019: NonNull=196,680 | MASTER=168,661 ( 85.75%) | DEREG=+16,644 ( 8.46%) | Combined=185,305 ( 94.22%) | Unmatched=11,375 (43 tails)
  2023: NonNull=150,278 | MASTER=145,917 ( 97.10%) | DEREG=+ 2,545 ( 1.69%) | Combined=148,462 ( 98.79%) | Unmatched= 1,816 (14 tails)
  2024: NonNull=159,842 | MASTER=157,959 ( 98.82%) | DEREG=+ 1,742 ( 1.09%) | Combined=159,701 ( 99.91%) | Unmatched=   141 (2 tails)
  2025: NonNull=180,924 | MASTER=180,648 ( 99.85%) | DEREG=+   275 ( 0.15%) | Combined=180,923 (100.00%) | Unmatched=     1 (1 tails)
  2026: NonNull=134,995 | MASTER=134,995 (100.00%) | DEREG=+     0 ( 0.00%) | Combined=134,9

In [7]:
# Table 5: Empirical Audit of Known Quirks (hhmm, '2400', Cancelled/Diverted Nulls, Airport ID vs IATA, FAA Formatting)
q = emp["quirks"]
print("=== Quirk 1: Local hhmm Leading-Zero Counts & 6 Departure Hour Blocks ===")
print("Leading-zero hhmm strings ('0001'..'0959'):", q["quirk_1_leading_zero_hhmm"])
print("ORD UA Departure Hour Block Distribution:")
for hb, cnt in sorted(q["quirk_1_hour_blocks"].items()):
    print(f"  {hb:18s}: {cnt:7,d} ({cnt/emp['otp']['total_ord_ua_rows']*100:5.2f}%)")

print("\n=== Quirk 2: Exact '2400' Time Value Occurrences & Verbatim Examples ===")
print("Counts by column:", q["quirk_2_2400_counts"])
for ex in q["quirk_2_2400_examples"][:4]:
    print("  Example:", ex)

print("\n=== Quirk 4: OriginAirportID vs Origin & DestAirportID vs Dest ===")
print("OriginAirportID -> Origin mapping:", q["quirk_4_orig_id_map"])
print(f"Unique ORD domestic destination IATA codes: {q['quirk_4_dest_count']} (Multi-ID collisions: {len(q['quirk_4_dest_multi_id'])})")

print("\n=== Quirk 6: FAA Registry UTF-8 BOM & Column Header Formatting ===")
for fn, fd in emp["faa"].items():
    print(f"  {fn:12s}: rows={fd['rows']:7,d}, raw_cols={fd['raw_col_count']:2d}, clean_cols={fd['clean_col_count']:2d}, has_utf8_bom={fd['has_bom']}, cols_preview={fd['columns'][:5]}")

=== Quirk 1: Local hhmm Leading-Zero Counts & 6 Departure Hour Blocks ===
Leading-zero hhmm strings ('0001'..'0959'): {'CRSDepTime': 201918, 'DepTime': 198584, 'CRSArrTime': 60363, 'ArrTime': 84082}
ORD UA Departure Hour Block Distribution:
  afternoon (14-17) : 159,868 (19.24%)
  early (05-08)     :  89,575 (10.78%)
  evening (17-20)   : 172,936 (20.81%)
  late (20-05)      :  90,011 (10.83%)
  midday (11-14)    : 132,822 (15.98%)
  morning (08-11)   : 185,729 (22.35%)

=== Quirk 2: Exact '2400' Time Value Occurrences & Verbatim Examples ===
Counts by column: {'ArrTime': 121, 'DepTime': 7, 'CRSArrTime': 1}
  Example: {'FlightDate': '2019-01-28', 'Operating_Airline': 'OO', 'Flight_Number': '5602', 'Origin': 'ORD', 'Dest': 'BNA', 'Tail_Number': 'N204SY', 'Column': 'ArrTime', 'CRSDepTime': '1950', 'DepTime': '2121', 'CRSArrTime': '2129', 'ArrTime': '2400'}
  Example: {'FlightDate': '2019-01-24', 'Operating_Airline': 'OO', 'Flight_Number': '5049', 'Origin': 'ORD', 'Dest': 'FWA', 'Tail_Num

In [8]:
# Table 1: TSA Daily Checkpoint Volumes (2019, 2023-2026) & BTS Lookups Verification
tsa = emp["tsa"]
print(f"=== TSA Checkpoint Volumes ({tsa['total_days']:,} unique days: {tsa['date_min']} .. {tsa['date_max']}) ===")
for yr, td in tsa["by_year"].items():
    print(f"  {yr} ({td['days']:3d} days, {td['min_date']}..{td['max_date']}): Mean={td['mean_throughput']:9,d} | Median={td['median_throughput']:9,d} | Min={td['min_throughput']:9,d} ({td['min_throughput_date']}) | Max={td['max_throughput']:9,d} ({td['max_throughput_date']})")

print("\n=== BTS Lookup Tables Summary ===")
for fn, ld in emp["lookups"].items():
    print(f"  {fn:24s}: {ld['row_count']:5,d} rows | {ld['bytes']:7,d} bytes | SHA-256={ld['sha256'][:16]}...")

=== TSA Checkpoint Volumes (1,777 unique days: 2019-01-01 .. 2026-11-12) ===
  2019 (365 days, 2019-01-01..2019-12-31): Mean=2,323,567 | Median=2,367,915 | Min=1,591,158 (2019-11-28) | Max=2,882,915 (2019-12-01)
  2023 (365 days, 2023-01-01..2023-12-31): Mean=2,352,187 | Median=2,402,206 | Min=1,534,786 (2023-01-31) | Max=2,908,785 (2023-11-26)
  2024 (366 days, 2024-01-01..2024-12-31): Mean=2,470,133 | Median=2,551,024 | Min=1,551,896 (2024-11-28) | Max=3,088,836 (2024-12-01)
  2025 (365 days, 2025-01-01..2025-12-31): Mean=2,484,208 | Median=2,532,054 | Min=1,559,165 (2025-11-27) | Max=3,134,613 (2025-11-30)
  2026 (316 days, 2026-01-01..2026-11-12): Mean=2,465,104 | Median=2,534,758 | Min=1,313,323 (2026-01-25) | Max=2,988,204 (2026-06-18)

=== BTS Lookup Tables Summary ===
  L_AIRCRAFT_TYPE.csv     :   453 rows |  14,177 bytes | SHA-256=4fdb4909c671582f...
  L_AIRPORT_ID.csv        : 6,918 rows | 332,844 bytes | SHA-256=cff27be77b3da705...
  L_CARRIER_HISTORY.csv   : 2,077 rows |  9